# BVID-FE Quickstart

This notebook mirrors [`01_empirical_quick.py`](01_empirical_quick.py) but adds annotated physics derivations and inline Plotly visualisations. It runs the **empirical tier** end-to-end on a 30-J impact against an IM7/8552 quasi-isotropic panel and shows the closed-form models that underpin BVID-FE.

For background on the modelling tiers and material library, see the [README](../README.md) (`Physics Models` and `Limitations` sections) and [`ARCHITECTURE.md`](../ARCHITECTURE.md).

Outputs are stripped on commit (`nbstripout examples/quickstart.ipynb`); re-execute the notebook locally to populate them.

In [ ]:
import numpy as np
import plotly.graph_objects as go

from bvidfe.analysis import AnalysisConfig, BvidAnalysis
from bvidfe.analysis.fe_mesh import build_fe_mesh
from bvidfe.core.geometry import ImpactorGeometry, PanelGeometry
from bvidfe.core.laminate import Laminate
from bvidfe.core.material import MATERIAL_LIBRARY
from bvidfe.impact.mapping import ImpactEvent
from bvidfe.impact.olsson import onset_energy, threshold_load
from bvidfe.sweep.parametric_sweep import sweep_energies
from bvidfe.viz.plotly_3d import mesh_damage_figure

## Material and laminate

We use the bundled IM7/8552 material card with a balanced quasi-isotropic stack `[0/45/-45/90]_s` at 0.152 mm per ply and a 150 x 100 mm panel. All lengths are millimetres; strengths are MPa; energies are joules.

In [ ]:
material = MATERIAL_LIBRARY["IM7/8552"]
layup = [0, 45, -45, 90, 90, -45, 45, 0]
ply_thickness_mm = 0.152
panel = PanelGeometry(Lx_mm=150, Ly_mm=100)
impact = ImpactEvent(
    energy_J=30.0,
    impactor=ImpactorGeometry(diameter_mm=16.0),
    mass_kg=5.5,
)
laminate = Laminate(material=material, layup_deg=layup, ply_thickness_mm=ply_thickness_mm)
print(f"material         : {material.name}")
print(f"layup (deg)      : {layup}")
print(f"panel (mm)       : {panel.Lx_mm} x {panel.Ly_mm}")
print(f"impact (J)       : {impact.energy_J}  (impactor d = {impact.impactor.diameter_mm} mm)")

## Olsson damage threshold

Damage onset is governed by the Olsson quasi-static threshold load, derived from a plate-bending / mode-II fracture energy balance:

$$P_c \;=\; \pi \sqrt{\dfrac{32\, D^{*}\, G_{IIc}}{3}}, \qquad D^{*} = \sqrt{D_{11} D_{22}\, \dfrac{A + 1}{2}}, \qquad A = \dfrac{D_{12} + 2 D_{66}}{\sqrt{D_{11} D_{22}}}$$

with $D^{*}$ Olsson's effective bending stiffness of the orthotropic laminate (from its CLT $D_{ij}$), and $G_{IIc}$ the mode-II interlaminar fracture toughness from the material card. The onset *energy* is then

$$E_{\mathrm{onset}} \;=\; \dfrac{k_b w_c^{2}}{2} + \dfrac{k_m w_c^{4}}{4} + \dfrac{P_c\, \delta_c}{n + 1}, \qquad k_b w_c + k_m w_c^{3} = P_c, \qquad P_c = K_c\, \delta_c^{\,n}$$

the energy the plate and the contact store at $P_c$. $k_b$ is the Navier-series point-load bending stiffness of the SSSS plate (with a boundary multiplier for clamped/free), $k_m$ the single-term von Kármán membrane stiffness that stiffens a plate deflecting more than its thickness, and $K_c \delta^n$ the Hertz contact law of the impactor tip ($n = 3/2$). For this thin 8-ply panel $P_c \approx 850$ N, the plate deflects about 2.6 thicknesses at $P_c$ (14 on the linear law) and $E_{\mathrm{onset}} \approx 0.84$ J (7.2 J without the membrane term), so about 29 J of the 30 J goes into the damage-area energy balance below; below the onset energy no damage is predicted and the knockdown is 1.

*References:* Olsson, R. (2001), *Composites Part A* **32**(9); Olsson, R. (2010), *Int. J. Solids Struct.* **47**(21); Timoshenko & Woinowsky-Krieger, *Theory of Plates and Shells*, ch. 4 and 13; Levy, S. (1942), NACA Report 737; Chia, C.Y. (1980), *Nonlinear Analysis of Plates*.

In [ ]:
Pc_N = threshold_load(laminate, panel, impact.impactor)
E_onset_J = onset_energy(laminate, panel, impact.impactor)
print(f"Olsson threshold load Pc : {Pc_N:7.1f} N")
print(f"Olsson onset energy      : {E_onset_J:7.2f} J")
print(f"Impact energy            : {impact.energy_J:7.2f} J  (above onset)")

## Peanut-template DPA distribution

Once the Olsson threshold is exceeded, a fraction $\alpha$ (`olsson_alpha`) of the energy above onset goes into mode-II delamination: $\mathrm{DPA} = \alpha\,(E - E_{\mathrm{onset}}) / G_{IIc}$, times impactor mass, shape and diameter factors that are 1 for the 5.5 kg, 16 mm hemispherical ASTM D7136 impactor. This projected damage area (DPA) is distributed across the ply interfaces using a *peanut template*. Each interface $i$ between plies of orientation $\theta_i$ and $\theta_{i+1}$ gets an elliptical delamination whose aspect ratio scales with the ply-angle jump,

$$\mathrm{AR}_i \;=\; 1 + 0.025\,\Delta\theta_i, \qquad \Delta\theta_i \in [0^\circ, 90^\circ]$$

where $\Delta\theta_i$ is the acute angle between the two fibre directions (angles are taken modulo 180°, so -45 and 135 are the same ply). The major axis lies along the fibres of the back-face-side ply $\theta_{i+1}$ (ply 0 is the impact face; Liu 1988), interfaces between plies of the same direction carry no delamination, and the per-interface size grows from the impact face toward the back face. A scalar multiplier is then solved (Brent's method) so that the polygon-union footprint of all ellipses equals the target DPA to within 1%. Every ellipse is kept inside the impacted panel (`impact_panel`, else `panel`) around the impact point: an ellipse that would cross the edge is shrunk, aspect ratio kept, and the union is re-solved for the same area. Here $\mathrm{DPA} \approx 2{,}500$ mm$^2$ over 6 interfaces (the two mid-plane 90° plies share a direction, so the interface between them carries none).

*Reference:* see `bvidfe.impact.shape_templates`; the peanut shape draws on the experimental observation that adjacent-ply mismatch governs delamination eccentricity (e.g. Olsson 2001, Bouvet *et al.* 2009).

## Soutis CAI knockdown

Compression-after-impact residual strength is predicted from a Soutis open-hole-equivalent knockdown,

$$\dfrac{\sigma_{\mathrm{CAI}}}{\sigma_{\mathrm{pristine}}} \;=\; \dfrac{1}{1 + k_s\left(\dfrac{\mathrm{DPA}}{A_{\mathrm{ref}}}\right)^{m}}$$

where $k_s$ and $m$ are material-specific empirical constants from the `OrthotropicMaterial` card (`soutis_k_s` = 4.8 and `soutis_m` = 0.5 by default), and $A_{\mathrm{ref}} = 15\,000$ mm$^2$ is the area of the 150 x 100 mm ASTM D7137 coupon on which $k_s$ is calibrated, whatever the size of the analysed panel. The pristine reference $\sigma_{\mathrm{pristine}}$ is the unnotched laminate strength from classical lamination theory (ply-discount last-ply failure, `bvidfe.failure.laminate_strength`).

*Reference:* Soutis, C., & Curtis, P. T. (1996), *Composites Sci. Technol.* **56**(6) 677-684.

## Whitney-Nuismer TAI knockdown

Tension-after-impact is treated as an equivalent circular open hole of radius $R = \sqrt{\mathrm{DPA}/\pi}$ with the point-stress criterion at a material characteristic distance $d_0$:

$$\dfrac{\sigma_N}{\sigma_{0}} \;=\; \dfrac{2}{2 + \xi^{2} + 3\xi^{4} - (K_{t\infty} - 3)\,(5\xi^{6} - 7\xi^{8})}, \qquad \xi \;=\; \dfrac{R}{R + d_0}$$

$K_{t\infty}$ is the infinite-plate stress concentration factor, with an orthotropic correction
$K_{t\infty} = 1 + \sqrt{2\,(\sqrt{E_x/E_y} - \nu_{xy}) + E_x/G_{xy}}$ (Lekhnitskii) that reduces to 3.0 for an isotropic plate.

*Reference:* Whitney, J. M., & Nuismer, R. J. (1974), *J. Composite Materials* **8**(3) 253-265.

## Run the empirical analysis

Expect a pristine strength of about 600 MPa, a residual strength of about 188 MPa (knockdown about 0.31) and a DPA of about 3,160 mm$^2$. The dent depth (0.61 mm) sits at its cap of half the laminate thickness at every energy for this thin panel.

In [ ]:
cfg = AnalysisConfig(
    material="IM7/8552",
    layup_deg=layup,
    ply_thickness_mm=ply_thickness_mm,
    panel=panel,
    impact=impact,
    loading="compression",
    tier="empirical",
)
result = BvidAnalysis(cfg).run()
print(result.summary())

## Inline Plotly visualisations

Three views of the result:

1. **Damage map** — top-down plan view of every per-interface delamination ellipse.
2. **Knockdown sweep** — Soutis CAI knockdown vs impact energy across 5-40 J.
3. **3D damage mesh** — hex-mesh boundary coloured by per-element damage factor (uses `bvidfe.viz.plotly_3d.mesh_damage_figure`).

In [ ]:
# 1. Damage map
fig_map = go.Figure()
fig_map.add_shape(
    type="rect", x0=0, y0=0, x1=panel.Lx_mm, y1=panel.Ly_mm,
    line=dict(color="black", width=1.5),
)
theta = np.linspace(0, 2 * np.pi, 100)
ifaces = sorted({d.interface_index for d in result.damage.delaminations})
for d in result.damage.delaminations:
    c = np.cos(np.radians(d.orientation_deg))
    s = np.sin(np.radians(d.orientation_deg))
    x_local = d.major_mm * np.cos(theta)
    y_local = d.minor_mm * np.sin(theta)
    x = c * x_local - s * y_local + d.centroid_mm[0]
    y = s * x_local + c * y_local + d.centroid_mm[1]
    fig_map.add_trace(go.Scatter(
        x=x, y=y, mode="lines", fill="toself", opacity=0.35,
        name=f"interface {d.interface_index}",
        line=dict(width=1),
    ))
fig_map.update_layout(
    title=f"BVID damage map (DPA = {result.dpa_mm2:.1f} mm^2, dent = {result.damage.dent_depth_mm:.2f} mm)",
    xaxis=dict(title="x [mm]", range=[0, panel.Lx_mm], scaleanchor="y", scaleratio=1),
    yaxis=dict(title="y [mm]", range=[0, panel.Ly_mm]),
    height=420,
    margin=dict(l=40, r=20, t=50, b=40),
)
fig_map.show()

In [ ]:
# 2. Knockdown sweep (empirical tier, 5..40 J)
sweep_df = sweep_energies(cfg, energies_J=[5, 10, 15, 20, 25, 30, 35, 40])
fig_kd = go.Figure()
fig_kd.add_trace(go.Scatter(
    x=sweep_df["energy_J"], y=sweep_df["knockdown"],
    mode="lines+markers", name="empirical",
))
fig_kd.update_layout(
    title="BVID empirical knockdown vs impact energy",
    xaxis_title="Impact energy [J]",
    yaxis_title="Knockdown (residual / pristine)",
    yaxis=dict(range=[0, 1.05]),
    height=420,
    margin=dict(l=40, r=20, t=50, b=40),
)
fig_kd.show()
sweep_df

In [ ]:
# 3. 3D damage mesh -- builds the hex mesh used by the fe3d tier and colours
# the boundary by per-element damage factor. Pure Plotly; no FE solve.
mesh = build_fe_mesh(cfg, result.damage)
fig_3d = mesh_damage_figure(mesh, title="BVID damage factor (3D hex mesh boundary)")
fig_3d.show()

## Modify and re-run

Edit the cell below to change the impact energy, layup, or material, then re-execute. The empirical tier runs in milliseconds so the loop is interactive. Material presets: `AS4/3501-6`, `AS4/8552`, `IM7/8552`, `T700/2510`, `T800/epoxy`, `T800S/3900-2B`. At 20 J the default inputs give a knockdown of about 0.41.

In [ ]:
# --- edit these ---
my_energy_J = 20.0
my_layup = [0, 45, -45, 90, 90, -45, 45, 0]
my_material = "IM7/8552"  # try "AS4/3501-6" too
# ------------------
my_cfg = AnalysisConfig(
    material=my_material,
    layup_deg=my_layup,
    ply_thickness_mm=0.152,
    panel=PanelGeometry(Lx_mm=150, Ly_mm=100),
    impact=ImpactEvent(
        energy_J=my_energy_J,
        impactor=ImpactorGeometry(diameter_mm=16.0),
        mass_kg=5.5,
    ),
    loading="compression",
    tier="empirical",
)
my_result = BvidAnalysis(my_cfg).run()
print(my_result.summary())